In [1]:
####################################
#ENVIRONMENT SETUP

In [2]:
#LIBRARIES
import os, sys

import numpy as np
import math

import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import xarray as xr

import pickle 

from tqdm import tqdm

In [3]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis","MPAS_Model_Data"))
from CLASSES_Directories import DirectoryManager_Class

In [4]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "InitialFigures")
dataType = "SurfaceVariableAnimations_Structured"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)

 DirectoryManager_Class Summary
 Main Directory:           /glade/u/home/aroseman/Projects/Regional-MPAS-Project
 Main Scratch Directory:   /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project
 Scratch Directory:        /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0
 Main Output Directory:    /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT
 Main Code Directory:      /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles
 Current Code Directory:   /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/DataAnalysis/MPAS_Model_Data/InitialFigures



In [5]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class,DataOperator_Class

In [6]:
RunType = ("TRACER","MOIST","NSSL")
SimulationTime = ("2022-06-30","2022-07-03")
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

Found 264/289 files matching expected times.
Opened history file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_NSSL/backup_RESTART2/history_cartesian/history.2022-06-30_00.00.00.latlon.nc
Opened diag file: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_NSSL/backup_RESTART2/diag_cartesian/diag.2022-06-30_00.00.00.latlon.nc

=== MPAS Structured (lat-lon) Model Data Summary ===
 Region:         TRACER
 Case:           MOIST
 Microphysics:   NSSL
 Resolution:     20-1km
 Time Step:      15mins
 Time Range:     2022-06-30 to 2022-07-03
 Coordinates:    ['latitude', 'longitude', 'nVertLevels', 'nVertLevelsP1', 'nSoilLevels']
 # History Files:264
 # Diag Files:   264
 # Time Steps:   264
 Data Directory: /glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.0/TRACER/MOIST/MPAS-Model_NSSL
 Static File:    TRACER_regional5250_scaled3_x20.8355

In [7]:
###############
#FUNCTIONS

In [33]:
def GetCLims_Average(ModelData, varNames, method="mean",
                     lower_pct=5, upper_pct=95):
    """
    Compute average or percentile-based (vmin, vmax) across all timesteps.
    Also stores the full list of per-timestep min/max values.
    Returns:
        climDictionary (summary dict[varName] = (vmin, vmax))
        full_climDictionary (detailed dict[varName] = {"vmins": [...], "vmaxs": [...]})
    """
    import numpy as np
    from tqdm import tqdm

    if isinstance(varNames, str):
        varNames = [varNames]

    # store lists of per-timestep values
    full_climDictionary = {v: {"vmins": [], "vmaxs": []} for v in varNames}

    for t in tqdm(range(len(ModelData.fileList)), desc="Processing timesteps"):
        data = ModelData.GetDataTimestep(t, printout=False)
        data_diag = ModelData.GetDataTimestep_diag(t, printout=False)

        for varName in varNames:
            variableSubset, _, _ = DataOperator_Class.GetVariable_Subset(
                ModelData, data, data_diag, ModelData.staticData, varName
            )
            vmin = variableSubset.min().item()
            vmax = variableSubset.max().item()
            full_climDictionary[varName]["vmins"].append(vmin)
            full_climDictionary[varName]["vmaxs"].append(vmax)

        data.close()
        data_diag.close()

    # summarize
    climDictionary = {}
    for varName in varNames:
        vmins = np.array(full_climDictionary[varName]["vmins"])
        vmaxs = np.array(full_climDictionary[varName]["vmaxs"])

        if method == "mean":
            climDictionary[varName] = (np.mean(vmins), np.mean(vmaxs))
        elif method == "percentile":
            climDictionary[varName] = (
                np.percentile(vmins, lower_pct),
                np.percentile(vmaxs, upper_pct)
            )
        elif method == "max":
            climDictionary[varName] = (
                np.min(vmins),
                np.max(vmaxs)
            )
        else:
            raise ValueError("method must be 'mean' or 'percentile'")

    return climDictionary, full_climDictionary

def LoadOrCreateCLims(ModelData, varNames, filePath="climDictionary2.pkl",
                              method="mean"):
    """
    Loads both the summarized and full clim dictionaries if they exist.
    Otherwise computes, saves, and returns them.
    Returns:
        climDictionary, full_climDictionary
    """
    import pickle
    import os

    if os.path.exists(filePath):
        print(f"Loading existing clim dictionaries from {filePath}")
        with open(filePath, "rb") as f:
            data = pickle.load(f)

        # backward compatibility: handle old format
        if isinstance(data, tuple):
            climDictionary, full_climDictionary = data
        else:
            climDictionary = data.get("climDictionary", {})
            full_climDictionary = data.get("full_climDictionary", {})
    else:
        print("File not found. Computing new clim dictionaries.")
        climDictionary, full_climDictionary = GetCLims_Average(
            ModelData, varNames, method=method
        )
        with open(filePath, "wb") as f:
            pickle.dump(
                {"climDictionary": climDictionary,
                 "full_climDictionary": full_climDictionary},
                f
            )
        print(f"Saved clim dictionaries to {filePath}")

    return climDictionary, full_climDictionary

In [25]:
#PlotVariable_with_Borders()

# Preload map features once
COAST = cfeature.COASTLINE.with_scale("50m")
BORDERS = cfeature.BORDERS.with_scale("50m")
STATES = cfeature.STATES.with_scale("50m")
LAND = cfeature.LAND.with_scale("50m")
LAKES = cfeature.LAKES.with_scale("50m")

def PlotVariable_with_Borders(variable, varName, lat,lon, multiplier,
                              outputFile=None, save=False, 
                              cmap="viridis", clim=(None,None), norm=None,
                              title=None, units=None,
                              center_colorbar=False):
    """
    Plot a uxarray or xarray variable on a map with coastlines, borders, and states,
    using Matplotlib (static PNG output). Works headlessly — no Selenium needed.
    """

    # Create figure
    fig, ax = plt.subplots(
        subplot_kw={'projection': ccrs.PlateCarree()},
        figsize=(9, 5)
    )

    num_levels=19
    levels = multiplier*np.linspace(clim[0],clim[1],num_levels)

    # if center_colorbar==True:
    #     cmap = "RdBu_r"
    #     vmax = max(abs(clim[0]), abs(clim[1]));  vmin = -vmax
    #     norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    #     levels = multiplier*np.linspace(vmin,vmax,num_levels)
    if center_colorbar==True:
        cmap = "RdBu_r"
        vmax = clim[1]; vmin = clim[0]
        norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
        levels = multiplier*np.linspace(vmin,vmax,num_levels)
    
    matrix = multiplier*variable.data
    # Scatter/contour fill (tricontourf works for unstructured grids)
    im = ax.contourf(
        lon, lat, matrix,
        levels=levels,
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree(),
    )

    # Add map features
    ax.add_feature(COAST, linewidth=1)
    ax.add_feature(BORDERS, linewidth=0.8)
    ax.add_feature(STATES, linewidth=0.5)
    ax.add_feature(LAND, facecolor="lightgray", alpha=0.3)
    ax.add_feature(LAKES, edgecolor="k", facecolor="none")

    # Colorbar
    if units is not None:
        label=varName +fr" (${units}$)"
    else: 
        label=varName
    plt.colorbar(im, ax=ax, orientation="vertical", label=label)

    #LABELS
    # Set extent to your data range (forces lat/lon ticks)
    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=ccrs.PlateCarree())
    
    # Add lat/lon ticks with degrees
    ax.set_xticks(np.linspace(lon.min(), lon.max(), 5), crs=ccrs.PlateCarree())
    ax.set_yticks(np.linspace(lat.min(), lat.max(), 5), crs=ccrs.PlateCarree())
    
    # # Format tick labels as degrees
    # lon_formatter = ccrs.LongitudeFormatter()
    # lat_formatter = ccrs.LatitudeFormatter()
    # ax.xaxis.set_major_formatter(lon_formatter)
    # ax.yaxis.set_major_formatter(lat_formatter)
    if title is not None:
        ax.set_title(title)
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")


    # Save or display
    if save:
        plt.savefig(outputFile, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved image: {outputFile}","\n")
        return None
    else:
        return fig

def SplitTimeString(timeString):
    date, time = timeString.split('_')
    time = time.replace('.', ':')
    return date,time

def GetUnits_Specific(varName):
    if "+" in varName:
        varName = varName.split("+")[0].strip()
        
    for d in (ModelData.unitsDictionary,
              ModelData.unitsDictionary_diag,
              ModelData.unitsDictionary_static):
        if varName in d:
            return d[varName]
    return None
                    
# #TESTING
# t=100
# data = ModelData.GetDataTimestep(t)
# data_diag = ModelData.GetDataTimestep_diag(t); print(f"\n")
# #defining variable names
# varNames = [
#     "q2"]
# # running
# variableDictionary = BuildVariableDictionary(varNames,data,data_diag,climDictionary)
# MakePlots(variableDictionary, save=False)

In [26]:
def GetVariableOutputFile(varName, t, ModelData, outputDirectory):
    folderName = f"{ModelData.region}_{ModelData.case}_{ModelData.mpType}/{varName}"
    timeString = ModelData.timeStrings[t]
    fileName = f"{varName}_{timeString}.png"
    filePath = DirectoryManager.GetOutputFile(outputDirectory, folderName, fileName)
    return filePath
    
def BuildVariableDictionary(varNames, dataSubset,dataSubset_diag,
                            lat,lon,climDictionary):
    variableDictionary = {}
    for varName in varNames:
        # print(f"Adding {varName}")

        # Getting Output File
        outputFilePath = GetVariableOutputFile(varName, t, ModelData, outputDirectory)
        
        # Handle addition of two variables
        if '+' in varName:
            var1, var2 = varName.split('+')
            var1 = var1.strip()
            var2 = var2.strip()
            
            subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset,
                                                          dataSubset_diag,dataSubset_static,var1)
            subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset,
                                                          dataSubset_diag,dataSubset_static,var2)
            variableSubset = subset1 + subset2

            # Combine CLims from both variables
            if (var1 in climDictionary) and (var2 in climDictionary):
                vmin = min(climDictionary[var1][0], climDictionary[var2][0])
                vmax = max(climDictionary[var1][1], climDictionary[var2][1])
                clim = (vmin, vmax)
        
        else:
            variableSubset = DataOperator_Class.GetData_Variable(ModelData, 
                                                                 dataSubset,dataSubset_diag,dataSubset_static,varName)
            clim = climDictionary[varName]

        #Setting up Units and Multiplier
        units = GetUnits_Specific(varName).replace(" ", r"\ ")
        if varName in ["qv","qc","qi","qr","q2","qfx"]:
            multiplier = 1/1e3
            units = units.replace('kg', 'g', 1)
        else:
            multiplier = 1

        #Setting up center_colorbar
        if varName in ['u10','v10','w','uReconstructZonal','uReconstructMeridional']:
            center_colorbar=True
        elif varName in ['hfx','qfx','lh']:
            center_colorbar=True
        else:
            center_colorbar=False
        
        # Store in dictionary
        variableDictionary[varName] = {
            "data": variableSubset,
            "lat": lat,
            "lon": lon,
            "units": units,
            "multiplier": multiplier,
            "outputFilePath": outputFilePath,
            "clim": clim,
            "center_colorbar": center_colorbar
        }
        
    return variableDictionary

def MakePlots(variableDictionary, save=False):
    date, time = SplitTimeString(ModelData.timeStrings[t])
    title = f"{ModelData.region}/{ModelData.case}/{ModelData.mpType} on {date} at {time}"
    
    for varName, contents in variableDictionary.items():
        # print(f"Plotting {varName}")
    
        data = contents["data"]
        lat  = contents["lat"]
        lon  = contents["lon"]
        units = contents["units"]
        multiplier = contents["multiplier"]
        outputFilePath = contents["outputFilePath"]
        clim = contents["clim"]
        center_colorbar = contents["center_colorbar"]
        
        fig = PlotVariable_with_Borders(data, varName, lat, lon, multiplier, 
                                        outputFilePath, save=save, 
                                        cmap="viridis", clim=clim,
                                        title=title, units=units,
                                        center_colorbar=center_colorbar)

In [27]:
#################
#RUNNING

In [62]:
#defining variable names
t=0
varNames = [
    "u10", "v10", "q2",
    "hfx", "qfx", "lh",
    "rainnc", "rainc",
    "refl10cm_1km",
    "greenfrac"
    ]

In [63]:
# climDictionary = GetCLims(ModelData,varNames) #run only once
climDictionary, _ = LoadOrCreateCLims(ModelData, varNames, filePath="climDictionary.pkl")

File not found. Computing new clim dictionaries.


Processing timesteps: 100%|██████████| 264/264 [00:18<00:00, 13.96it/s]

Saved clim dictionaries to climDictionary.pkl


In [66]:
#defining variable names
t=0
varNames = [
    "u10", "v10", "q2",
    "hfx", "qfx", "lh",
    "rainnc+rainc",
    "refl10cm_1km"
] + (["greenfrac"] if t == 0 else [])

In [67]:
#running
num_times = ModelData.Ntime
for count, t in enumerate(tqdm(range(num_times), desc="Processing timesteps")):
    if t % 10 == 0: print(f"Currently working on time {t}/{num_times}","\n")

    #Loading Data
    [dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _, _] = DataOperator_Class.GetData_Subset(ModelData,t)

    if (count == 1) and ("greenfrac" in varNames):
        varNames.remove("greenfrac")
    
    # runningx
    variableDictionary = BuildVariableDictionary(varNames,dataSubset,dataSubset_diag,
                                                 lat,lon,climDictionary)
    MakePlots(variableDictionary, save=True)

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/q2/q2_2022-06-30_20.30.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/hfx/hfx_2022-06-30_20.30.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/qfx/qfx_2022-06-30_20.30.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/lh/lh_2022-06-30_20.30.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structu

Processing timesteps:  31%|███▏      | 83/264 [06:17<15:22,  5.10s/it]

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/refl10cm_1km/refl10cm_1km_2022-06-30_20.30.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/u10/u10_2022-06-30_20.45.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/v10/v10_2022-06-30_20.45.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/q2/q2_2022-06-30_20.45.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariab

Processing timesteps:  32%|███▏      | 84/264 [06:22<15:15,  5.08s/it]

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/refl10cm_1km/refl10cm_1km_2022-06-30_20.45.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/u10/u10_2022-06-30_21.00.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/v10/v10_2022-06-30_21.00.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/q2/q2_2022-06-30_21.00.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariab

...

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/refl10cm_1km/refl10cm_1km_2022-07-01_06.00.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/u10/u10_2022-07-01_06.15.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/v10/v10_2022-07-01_06.15.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/q2/q2_2022-07-01_06.15.00.png 

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariab

Processing timesteps:  46%|████▌     | 122/264 [09:25<10:53,  4.60s/it]

Saved image: /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/OUTPUT/DataAnalysis/MPAS_Model_Data/InitialFigures/SurfaceVariableAnimations_Structured/TRACER_MOIST_NSSL/refl10cm_1km/refl10cm_1km_2022-07-01_06.15.00.png 



Processing timesteps:  46%|████▌     | 122/264 [09:25<10:58,  4.64s/it]

KeyboardInterrupt



<Figure size 900x500 with 0 Axes>

In [ ]:
#################
#MAKING ANIMATION

In [ ]:
#Needed Libraries
# from matplotlib.animation import FuncAnimation, PillowWriter
# from PIL import Image

# from moviepy import VideoFileClip, vfx

#Importing AnimationPlotting_Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import AnimationPlotting_Class

In [ ]:
def GetPlottingFileName(varName,outputDirectory,ModelData):
    plottingFileName = f"{varName}.gif"
    plottingFilePath = DirectoryManager.GetOutputFile(outputDirectory, 
                                                      f"{ModelData.region}_{ModelData.case}_{ModelData.mpType}/{varName}", 
                                                      plottingFileName)
    return plottingFilePath

In [ ]:
# getting ideal fps
fps = AnimationPlotting_Class.CalculateFPS(num_frames=ModelData.Ntime, time_interval_minutes=15, desired_duration_min=1)

In [ ]:
# running animation
for varName in varNames:
    if varName=="greenfrac": continue
    print(f"Working on {varName}","\n")

    # Setting up output file
    plottingFilePath = GetPlottingFileName(varName,outputDirectory,ModelData)
    AnimationPlotting_Class.CreateAnimation(ModelData, DirectoryManager,
                                            outputDirectory, plottingFilePath, GetVariableOutputFile,
                                            varName, start_t=0, end_t=ModelData.Ntime,
                                            fps=2)

In [ ]:
#converting animation to mp4
for varName in varNames:
    if varName=="greenfrac": continue
    input_file = GetPlottingFileName(varName,outputDirectory,ModelData)
    output_file = input_file.replace(".gif", ".mp4")
    AnimationPlotting_Class.convertGIFtoMP4(input_file, output_file,fps=fps)